# 04 — Mutation Stability Scoring

In this step, I assign a rule-based stability score to each mutation.

The scoring is based on:
- residue environment (core vs surface)
- mutation type (conservative, moderate, disruptive)

The goal is to classify mutations as:
- stabilizing
- neutral
- destabilizing

This provides an interpretable first-pass evaluation before using physics-based tools like FoldX.

***Imports***

In [1]:
import pandas as pd

***Load mutation table***

In [2]:
df = pd.read_csv("../data/processed/mutation_panel.csv")
df.head()

,mutation_id,position,resname_3,resname_1,wt,mut,environment_group,environment,ss_simple,rsa,mutation_class,rationale
0,L8I,8,LEU,L,L,I,core,core,helix,0.0,conservative,Maintains hydrophobic character with similar s...
1,L8A,8,LEU,L,L,A,core,core,helix,0.0,moderate,Reduces side-chain volume and may create a cav...
2,L8D,8,LEU,L,L,D,core,core,helix,0.0,disruptive,Introduces a charged residue into the buried c...
3,V29I,29,VAL,V,V,I,core,core,helix,0.0,conservative,Retains hydrophobic chemistry with a similar b...
4,V29A,29,VAL,V,V,A,core,core,helix,0.0,moderate,Shrinks side-chain volume and may weaken tight...


***Define scoring rules - Base score from mutation class***

In [3]:
def mutation_class_score(mutation_class):
    if mutation_class == "conservative":
        return +1
    elif mutation_class == "moderate":
        return -1
    elif mutation_class == "disruptive":
        return -3

***Environment modifier***

In [4]:
def environment_modifier(env):
    if env == "core":
        return 2   # amplify effects
    elif env == "surface":
        return 1   # weaker effect
    else:
        return 1

***Compute stability score***

In [5]:
def compute_stability_score(row):
    base = mutation_class_score(row["mutation_class"])
    modifier = environment_modifier(row["environment_group"])
    return base * modifier

df["stability_score"] = df.apply(compute_stability_score, axis=1)

df.head()

,mutation_id,position,resname_3,resname_1,wt,mut,environment_group,environment,ss_simple,rsa,mutation_class,rationale,stability_score
0,L8I,8,LEU,L,L,I,core,core,helix,0.0,conservative,Maintains hydrophobic character with similar s...,2
1,L8A,8,LEU,L,L,A,core,core,helix,0.0,moderate,Reduces side-chain volume and may create a cav...,-2
2,L8D,8,LEU,L,L,D,core,core,helix,0.0,disruptive,Introduces a charged residue into the buried c...,-6
3,V29I,29,VAL,V,V,I,core,core,helix,0.0,conservative,Retains hydrophobic chemistry with a similar b...,2
4,V29A,29,VAL,V,V,A,core,core,helix,0.0,moderate,Shrinks side-chain volume and may weaken tight...,-2


***Convert score to label***

In [6]:
def classify_effect(score):
    if score >= 1:
        return "stabilizing"
    elif score == 0:
        return "neutral"
    else:
        return "destabilizing"

df["predicted_effect"] = df["stability_score"].apply(classify_effect)

df

,mutation_id,position,resname_3,resname_1,wt,mut,environment_group,environment,ss_simple,rsa,mutation_class,rationale,stability_score,predicted_effect
0,L8I,8,LEU,L,L,I,core,core,helix,0.000000,conservative,Maintains hydrophobic character with similar s...,2,stabilizing
1,L8A,8,LEU,L,L,A,core,core,helix,0.000000,moderate,Reduces side-chain volume and may create a cav...,-2,destabilizing
2,L8D,8,LEU,L,L,D,core,core,helix,0.000000,disruptive,Introduces a charged residue into the buried c...,-6,destabilizing
3,V29I,29,VAL,V,V,I,core,core,helix,0.000000,conservative,Retains hydrophobic chemistry with a similar b...,2,stabilizing
4,V29A,29,VAL,V,V,A,core,core,helix,0.000000,moderate,Shrinks side-chain volume and may weaken tight...,-2,destabilizing
5,V29D,29,VAL,V,V,D,core,core,helix,0.000000,disruptive,Introduces a charged residue into a buried hyd...,-6,destabilizing
6,L56I,56,LEU,L,L,I,core,core,loop,0.000000,conservative,Maintains hydrophobic character with similar s...,2,stabilizing
7,L56A,56,LEU,L,L,A,core,core,loop,0.000000,moderate,Reduces side-chain volume and may create a cav...,-2,destabilizing
8,L56D,56,LEU,L,L,D,core,core,loop,0.000000,disruptive,Introduces a charged residue into the buried c...,-6,destabilizing
9,D18E,18,ASP,D,D,E,surface,surface,loop,0.343558,conservative,Retains negative charge while slightly extendi...,1,stabilizing


***Sort mutations***

In [7]:
df_sorted = df.sort_values("stability_score", ascending=False)
df_sorted

,mutation_id,position,resname_3,resname_1,wt,mut,environment_group,environment,ss_simple,rsa,mutation_class,rationale,stability_score,predicted_effect
0,L8I,8,LEU,L,L,I,core,core,helix,0.000000,conservative,Maintains hydrophobic character with similar s...,2,stabilizing
6,L56I,56,LEU,L,L,I,core,core,loop,0.000000,conservative,Maintains hydrophobic character with similar s...,2,stabilizing
3,V29I,29,VAL,V,V,I,core,core,helix,0.000000,conservative,Retains hydrophobic chemistry with a similar b...,2,stabilizing
15,Q41N,41,GLN,Q,Q,N,surface,surface,loop,0.606061,conservative,Retains polar amide chemistry with a slightly ...,1,stabilizing
12,R73K,73,ARG,R,R,K,surface,surface,loop,0.625000,conservative,Retains positive charge with similar surface e...,1,stabilizing
9,D18E,18,ASP,D,D,E,surface,surface,loop,0.343558,conservative,Retains negative charge while slightly extendi...,1,stabilizing
10,D18N,18,ASP,D,D,N,surface,surface,loop,0.343558,moderate,Removes formal charge but preserves polar hydr...,-1,destabilizing
13,R73Q,73,ARG,R,R,Q,surface,surface,loop,0.625000,moderate,Removes positive charge while keeping a polar ...,-1,destabilizing
16,Q41E,41,GLN,Q,Q,E,surface,surface,loop,0.606061,moderate,Introduces a negative charge at a previously n...,-1,destabilizing
7,L56A,56,LEU,L,L,A,core,core,loop,0.000000,moderate,Reduces side-chain volume and may create a cav...,-2,destabilizing


***Quick summary***